In [ ]:
import sys
from pathlib import Path

# 1. Определяем корень проекта
# (подбираем количество .parent, чтобы попасть в max_projects)
project_root = Path.cwd().parent

# 2. До,авляем корень в пути поиска модулей
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 3. Проверяем, что путь до,авлен
print(f"✅ Project root: {project_root}")
print(f"✅ Sys path: {sys.path[:3]}...")

import time
import json
import random
from curl_cffi import requests

# --- НАСТРОЙКИ СКРИПТА ---
# Актуальный список артикулов для проверки
PRODUCT_LIST = [
    189954586
    489570061,
    489570482,
    489570201,
    489570288,
    209667566,
    189579546,
    209666680,
    209664375,
    489569911,
    489570587,
    489569910,
    189956071,
    489570390,
    209667880,
    171386618,
    189956512,
    189955657,
    489570060,
    189956346,
    154791822,
    148565424,
    489570481,
    147675447,
    160163535,
    272584722,
    417425678,
    489570200,
    489570586,
    189955217,
    153272519,
    471231778,
    467180717,
    417425261,
    489570287,
    471230169,
    161217712,
    161217369,
    154790374,
    471238078,
    467175081,
    234906388,
    189954913,
    153272594,
    272629524,
    544870425,
    235754763,
    235759561,
    348509358,
    348514902,
    235765421,
    272626990,
    417428035,
    175012432,
    165567400,
    189917678,
    168103192,
    365759091,
    160165106,
]
# Время ожидания между запусками парсера (3600 секунд = 1 час)
UPDATE_INTERVAL = 3600 

# Новые куки, которые ты скопировал из Google Chrome
RAW_COOKIES = (
    "external-locale=ru; _wbauid=2477851001764666101; "
    "x-supplier-id-external=11ab4eb4-7970-46fa-bee0-d2f552620e8a; "
    "cfidsw-wb=pKF0dyplztYFHsrWM52CB0N5POPzetZFp/och+l+rkWy/y3oCRCbsdHV7vusHD9eFP8UGr/K8k97bawlCxiwqnsyCm0g7WTdywRJ8DSsBCVMeukjfwjf9R02557auDiZtRpBrnUwp6Ws/9imc4hClvAE8SlEagmF90aQon0=; "
    "__zzatw-wb=MDA0dC0yYBwREFsKEH49WgsbSl1pCENQGC9LXz1uLWEPJ3wjYnwgGWsvC1RDMmUIPkBNOTM5NGZwVydgTmAkS1ZVfycdEndtH0FLVCNyM3dlaXceViUTFmcPRyJ1F0hAGxI6aCU6f1JpGWUzDldjGAsmVDVfP3wnHxZ3byxScX9NfXY3PmJ+MQ9pOSRjCh9+OFoLDWk3XBQ8dWU+SHR4MTxtI2FLWh9EUT9FbllGaXUVF0M8HHsNKkNtLToZUXYQQlh4cBpEN0AYfxVZUnUpbn06MBtFVx0YTF4jQw8JfyciQ3skKVQ4EmNudnN1Lz8eURp7FiJER0lrZU5TQixmG3EVTQgNND1aciIPWzklWAgSPwsmIBh8cyRXDQ1fPkFubxt/Nl0cOWMRCxl+OmNdRkc3FSR7dSYKCTU3YnAvTCB7SykWRxsyYV5GaXUVAg8FUF2J3cmQm4kX0RdJXkOSjNaThF0JyUNDBBdQ0NtMDFsVxlRDxZhDhYYRRcje0I3Yhk4QhgvPV8/YngiD2lIYCZIXU4JKxsVeXEpS3FPLH12X30beylOIA0lVBMhP05yxKmKfQ==; "
    "x_wbaas_token=1.1000.cecb542ad0c148a8b9fb4e89f978a0d2.MTV8MTUwLjI0MS43Ni40NXxNb3ppbGxhLzUuMCAoV2luZG93cyBOVCAxMC4wOyBXaW42NDsgeDY0KSBBcHBsZVdlYktpdC81MzcuMzYgKEtIVE1MLCBsaWtlIEdlY2tvKSBDaHJvbWUvMTQ3LjAuMC4wIFNhZmFyaS81MzcuMzZ8MTc3OTQzNTk3NHxyZXVzYWJsZXwyfGV5Sm9ZWE5vSWpvaUluMD18MHwzfDE3NzkzMDYzNzR8MQ==.MEUCIQD51iDwxL0Hm3RESPKuXl+3q14WD1+OPrj58f969iV+igIgfcxgqoSLj4XhkizsV1ZWvg7zX6D9NSa8hEvJCS4IavY="
)




def get_wb_product_data(session, item_id):
    """
    Выполняет прямой сетевой запрос к внутреннему API Wildberries.
    Использует рабочие куки Chrome и стабильный TLS-отпечаток.
    """
    url = "https://www.wildberries.ru/__internal/card/cards/v4/detail"
    
    # Входящие параметры запроса (ID товара, валюта и базовый складской регион)
    params = {
        "appType": 1,
        "curr": "rub",
        "dest": "-1257786",
        "nm": item_id
    }
    
    # Заголовки запроса. Значение User-Agent мы взяли из твоего токена безопасности
    headers = {
        "Accept": "*/*",
        "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8",
        "Connection": "keep-alive",
        "Cookie": RAW_COOKIES,
        "Origin": "https://www.wildberries.ru",
        "Referer": f"https://www.wildberries.ru/catalog/{item_id}/detail.aspx",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36"
    }

    try:
        # Делаем запрос через chrome110 — этот отпечаток точно поддерживается твоей системой
        response = session.get(url, params=params, headers=headers, timeout=15)
        
        # Если Wildberries принял наши куки и ответил успешно (код 200)
        if response.status_code == 200:
            return response.json()
        else:
            return {"error": f"Сервер вернул код ошибки: {response.status_code}."}
            
    except Exception as e:
        return {"error": f"Ошибка сети при запросе: {str(e)}"}


def parse_all_products(products):
    """
    Циклически обходит список всех товаров, используя одну сессию.
    """
    results = {}
    
    # Создаем сессию с гарантированно поддерживаемым отпечатком Chrome 110
    with requests.Session(impersonate="chrome110") as session:
        
        for index, item_id in enumerate(products, 1):
            print(f"[{index}/{len(products)}] Получение данных для товара: {item_id}")
            
            # Отправляем запрос к API
            data = get_wb_product_data(session, item_id)
            
            # Проверяем, успешный ли ответ или вернулась ошибка
            if "error" not in data:
                results[item_id] = data
                print(f"   ✅ Данные успешно получены!")
            else:
                results[item_id] = data
                print(f"   ❌ Ошибка: {data['error']}")
            
            # Пауза от 1.5 до 3 секунд между товарами для имитации человека
            time.sleep(random.uniform(1.5, 3.0))
            
    return results


def save_results(data):
    """Сохраняет собранный словарь в JSON-файл."""
    filename = "wb_products_data.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    print(f"Файл {filename} успешно обновлен.\n")


# --- ГЛАВНЫЙ БЛОК ЗАПУСКА ---
if __name__ == "__main__":
    print("=== Chrome-парсер WB на новых куках запущен ===")
    

    current_time = time.strftime('%H:%M:%S')
    print(f"\n--- Старт цикла сбора данных ({current_time}) ---")

    # Собираем данные
    final_data = parse_all_products(PRODUCT_LIST)

    # Записываем результат в файл
    save_results(final_data)

    print(f"Ожидание следующего часа для обновления данных...")
    time.sleep(UPDATE_INTERVAL)

✅ Project root: c:\Users\123\Desktop\WildPositionMonitor
✅ Sys path: ['c:\\Users\\123\\Desktop\\WildPositionMonitor', 'C:\\Python314\\python314.zip', 'C:\\Python314\\DLLs']...


In [13]:
from services.wb_service import WildberriesService
import aiohttp

In [ ]:
from curl_cffi.requests import AsyncSession
from services.wb_service import WildberriesService
import asyncio

# Создаем "улучшенную" версию запроса прямо для теста
async def bypass_test(article_id):
    # Используем имитацию Chrome
    async with AsyncSession(impersonate="chrome120") as s:
        url = "https://card.wb.ru/cards/v4/detail"
        params = {
            "appType": 1,
            "curr": "rub",
            "dest": -1257786,
            "spp": 30,
            "nm": article_id
        }
        
        headers = {
            'Accept': '*/*',
            'Accept-Language': 'ru-RU,ru;q=0.9',
            'Origin': 'https://www.wildberries.ru',
            'Referer': f'https://www.wildberries.ru/catalog/{article_id}/detail.aspx'
        }

        print(f"🚀 Про,уем о,ход через curl_cffi для {article_id}...")
        response = await s.get(url, params=params, headers=headers)
        
        if response.status_code == 200:
            data = response.json()
            products = data.get('products', [])
            if products:
                product = products[0]
                name = product.get('name')
                price = product.get('sizes')[0].get('price', {}).get('product', 0) // 100
                print(f"✅ Успех! Товар: {name}")
                print(f"💰 Цена: {price} ру,.")
            else:
                print("⚠️ Ответ пустой (products не найден)")
        elif response.status_code == 403:
            print("❌ Все еще 403. Похоже, твой IP попал во временный ,ан.")
        else:
            print(f"❓ Статус: {response.status_code}")

await bypass_test(771133983)

🚀 Пробуем обход через curl_cffi для 771133983...
✅ Успех! Товар: Горелка газовая с пьезоподжигом туристическая
💰 Цена: 160 руб.


=== Chrome-парсер WB на новых куках запущен ===

--- Старт цикла сбора данных (12:14:50) ---
[1/17] Получение данных для товара: 189954586
   ✅ Данные успешно получены!
[2/17] Получение данных для товара: 489570061
   ✅ Данные успешно получены!
[3/17] Получение данных для товара: 489570482
   ✅ Данные успешно получены!
[4/17] Получение данных для товара: 489570201
   ✅ Данные успешно получены!
[5/17] Получение данных для товара: 489570288
   ✅ Данные успешно получены!
[6/17] Получение данных для товара: 209667566
   ✅ Данные успешно получены!
[7/17] Получение данных для товара: 189579546
   ✅ Данные успешно получены!
[8/17] Получение данных для товара: 209666680
   ✅ Данные успешно получены!
[9/17] Получение данных для товара: 209664375
   ✅ Данные успешно получены!
[10/17] Получение данных для товара: 489569911
   ✅ Данные успешно получены!
[11/17] Получение данных для товара: 489570587
   ✅ Данные успешно получены!
[12/17] Получение данных для товара: 489569910
   ✅ Данные успешно по